# 第18章 人工智能 - Part 3: 神经网络与深度学习
# Chapter 18 AI - Part 3: Neural Networks and Deep Learning

**Cambridge A-Level Computer Science (9618)**

---

## 本节学习目标 (Learning Objectives)

| 编号 | 目标 | 关键术语 |
|------|------|----------|
| 1 | 理解人工神经网络的结构 | ANN, Input Layer, Hidden Layer, Output Layer |
| 2 | 掌握单个神经元的计算过程 | Weights, Biases, Activation Function |
| 3 | 理解前向传播 | Forward Propagation |
| 4 | 了解反向传播的基本思想 | Back Propagation, Gradient Descent |
| 5 | 了解深度学习和CNN的概念 | Deep Learning, CNN |

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)



## 1. 从生物神经元到人工神经元

### 1.1 生物神经元

人脑有约**860亿个神经元**，每个神经元通过**突触(Synapse)**与其他神经元连接。

生物神经元的工作方式：
1. 通过**树突(Dendrites)**接收多个输入信号
2. 在**细胞体(Cell Body)**中整合信号
3. 如果信号总和超过**阈值(Threshold)**，通过**轴突(Axon)**发出输出信号

### 1.2 人工神经元（感知器 Perceptron）

模拟生物神经元：
1. 接收多个**输入(Inputs)** $x_1, x_2, ..., x_n$
2. 每个输入乘以一个**权重(Weight)** $w_1, w_2, ..., w_n$
3. 计算加权和，加上**偏置(Bias)** $b$
4. 通过**激活函数(Activation Function)**产生输出

$$z = \sum_{i=1}^{n} w_i \cdot x_i + b$$
$$\text{output} = \sigma(z)$$

### 为什么需要权重和偏置？

- **权重(Weights)**：决定每个输入的重要程度。权重大 = 该输入对结果影响大
- **偏置(Bias)**：让神经元即使所有输入为0也能产生输出。相当于线性函数的截距

### 为什么需要激活函数？

如果没有激活函数，多层网络就是多个线性变换的组合，结果还是线性的！

激活函数引入**非线性**，让网络能学习复杂的模式。这是神经网络强大的关键！

In [ ]:
# === 可视化：单个神经元的结构 ===

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('人工神经元 (Artificial Neuron) 结构', fontsize=16, fontweight='bold')

# 输入节点
inputs = [('x1', 0.05, 0.8), ('x2', 0.05, 0.5), ('x3', 0.05, 0.2)]
weights = ['w1=0.5', 'w2=-0.3', 'w3=0.8']

for (name, x, y), w in zip(inputs, weights):
    circle = plt.Circle((x+0.05, y), 0.04, color='#3498DB', ec='black', lw=2, zorder=5)
    ax.add_patch(circle)
    ax.text(x+0.05, y, name, ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    # 连线到求和节点
    ax.annotate('', xy=(0.35, 0.5), xytext=(x+0.09, y),
               arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=1.5))
    # 权重标签
    mx = (x+0.09 + 0.35) / 2
    my = (y + 0.5) / 2
    ax.text(mx-0.02, my+0.03, w, fontsize=9, color='#E74C3C', fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.15', facecolor='lightyellow', alpha=0.9))

# 求和节点
sum_circle = plt.Circle((0.4, 0.5), 0.06, color='#F39C12', ec='black', lw=2, zorder=5)
ax.add_patch(sum_circle)
ax.text(0.4, 0.5, 'Sum', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(0.4, 0.38, 'z = sum(wi*xi) + b', ha='center', fontsize=9, color='navy', style='italic')

# 偏置
bias_circle = plt.Circle((0.4, 0.75), 0.03, color='#9B59B6', ec='black', lw=2, zorder=5)
ax.add_patch(bias_circle)
ax.text(0.4, 0.75, 'b', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
ax.annotate('', xy=(0.4, 0.56), xytext=(0.4, 0.72),
           arrowprops=dict(arrowstyle='->', color='#9B59B6', lw=1.5))
ax.text(0.45, 0.68, 'bias=0.1', fontsize=9, color='#9B59B6')

# 激活函数
ax.annotate('', xy=(0.6, 0.5), xytext=(0.46, 0.5),
           arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=2))

act_rect = plt.Rectangle((0.6, 0.42), 0.15, 0.16, facecolor='#2ECC71',
                         edgecolor='black', lw=2, zorder=5)
ax.add_patch(act_rect)
ax.text(0.675, 0.5, 'Activation\nFunction', ha='center', va='center',
        fontsize=10, fontweight='bold', color='white')
ax.text(0.675, 0.36, 'sigma(z)', ha='center', fontsize=9, color='#27AE60', style='italic')

# 输出
ax.annotate('', xy=(0.88, 0.5), xytext=(0.75, 0.5),
           arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=2))
out_circle = plt.Circle((0.92, 0.5), 0.04, color='#E74C3C', ec='black', lw=2, zorder=5)
ax.add_patch(out_circle)
ax.text(0.92, 0.5, 'y', ha='center', va='center', fontsize=14, fontweight='bold', color='white')
ax.text(0.92, 0.42, 'Output', ha='center', fontsize=10, color='#C0392B')

# 计算示例
ax.text(0.5, 0.12, 'Example: z = 0.5*1 + (-0.3)*0 + 0.8*1 + 0.1 = 1.4',
        ha='center', fontsize=11, color='navy',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))
ax.text(0.5, 0.04, 'output = sigmoid(1.4) = 0.802',
        ha='center', fontsize=11, color='#C0392B',
        bbox=dict(boxstyle='round', facecolor='mistyrose', edgecolor='red'))

plt.tight_layout()


In [ ]:
# === 常见激活函数对比 ===

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def tanh(x):
    return np.tanh(x)

def step(x):
    return np.where(x >= 0, 1, 0)

x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('常见激活函数 (Activation Functions)', fontsize=16, fontweight='bold')

funcs = [
    (step, 'Step Function\n阶跃函数', '#E74C3C',
     'f(x) = 0 if x<0, 1 if x>=0\n最简单，但不可微分，无法用梯度下降'),
    (sigmoid, 'Sigmoid Function\nS形函数', '#3498DB',
     'f(x) = 1/(1+e^-x)\n输出在(0,1)，适合概率。但有梯度消失问题'),
    (tanh, 'Tanh Function\n双曲正切', '#2ECC71',
     'f(x) = tanh(x)\n输出在(-1,1)，以0为中心。也有梯度消失'),
    (relu, 'ReLU Function\n线性整流', '#F39C12',
     'f(x) = max(0,x)\n计算简单高效，目前最常用！但有"死神经元"问题'),
]

for idx, (func, title, color, desc) in enumerate(funcs):
    ax = axes[idx // 2][idx % 2]
    y = func(x)
    ax.plot(x, y, color=color, lw=3)
    ax.axhline(y=0, color='gray', lw=0.5, ls='--')
    ax.axvline(x=0, color='gray', lw=0.5, ls='--')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.02, desc, transform=ax.transAxes, fontsize=8,
            verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()
print('考试重点：Sigmoid用于输出层（二分类），ReLU用于隐藏层（现代标准）。')


## 2. 神经网络结构

### 2.1 多层网络

单个神经元只能学习线性分割。把多个神经元组织成**层(Layer)**，再把多层连接起来，就构成了**人工神经网络(ANN)**。

三种层：
- **输入层(Input Layer)**：接收原始数据，不做计算
- **隐藏层(Hidden Layer)**：做计算的中间层，可以有多个
- **输出层(Output Layer)**：产生最终结果

### 2.2 全连接层 (Fully Connected / Dense Layer)

每一层的每个神经元都与下一层的**所有**神经元连接。

### 为什么需要隐藏层？

隐藏层让网络能学习**中间表示(Intermediate Representations)**：
- 第一层可能学习简单特征（如边缘、角度）
- 中间层组合简单特征形成复杂特征（如眼睛、鼻子）
- 最后一层用复杂特征做决策（如：这是一张人脸）

**万能近似定理(Universal Approximation Theorem)**：一个有足够多神经元的单隐藏层网络可以近似任何连续函数。但实际上，多个较小的隐藏层通常比一个巨大的隐藏层更有效。

In [ ]:
# === 可视化：神经网络结构 ===

def draw_neural_network(ax, layer_sizes, title=''):
    '''绘制神经网络结构图'''
    n_layers = len(layer_sizes)
    max_neurons = max(layer_sizes)
    layer_colors = ['#3498DB', '#F39C12', '#F39C12', '#E74C3C']
    layer_names = ['Input\n输入层', 'Hidden 1\n隐藏层1', 'Hidden 2\n隐藏层2', 'Output\n输出层']

    # 确保颜色和名称匹配层数
    if n_layers == 3:
        layer_colors = ['#3498DB', '#F39C12', '#E74C3C']
        layer_names = ['Input\n输入层', 'Hidden\n隐藏层', 'Output\n输出层']

    v_spacing = 0.8 / max_neurons
    h_spacing = 0.8 / (n_layers - 1)

    # 画连接线
    for i in range(n_layers - 1):
        for j in range(layer_sizes[i]):
            for k in range(layer_sizes[i+1]):
                x1 = 0.1 + i * h_spacing
                y1 = 0.5 + (j - (layer_sizes[i]-1)/2) * v_spacing
                x2 = 0.1 + (i+1) * h_spacing
                y2 = 0.5 + (k - (layer_sizes[i+1]-1)/2) * v_spacing
                ax.plot([x1, x2], [y1, y2], color='#BDC3C7', lw=0.5, alpha=0.5, zorder=1)

    # 画神经元
    for i in range(n_layers):
        x = 0.1 + i * h_spacing
        for j in range(layer_sizes[i]):
            y = 0.5 + (j - (layer_sizes[i]-1)/2) * v_spacing
            circle = plt.Circle((x, y), 0.025, color=layer_colors[i],
                              ec='black', lw=1.5, zorder=5)
            ax.add_patch(circle)
        # 层标签
        ax.text(x, 0.05, layer_names[i], ha='center', va='center',
               fontsize=9, fontweight='bold')
        ax.text(x, 0.95, f'{layer_sizes[i]}个', ha='center', fontsize=9, color='gray')

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')


fig, axes = plt.subplots(1, 3, figsize=(18, 6))

draw_neural_network(axes[0], [3, 4, 1], '简单网络\n(3-4-1)')
draw_neural_network(axes[1], [4, 5, 3, 2], '深层网络\n(4-5-3-2)')
draw_neural_network(axes[2], [2, 4, 4, 1], 'XOR问题网络\n(2-4-4-1)')

plt.tight_layout()
plt.show()
print('每条线代表一个权重参数。网络越大，参数越多，能学习的模式越复杂。')
print(f'左图参数数量: {3*4+4}(权重) + {4+1}(偏置) = {3*4+4+4+1}')


## 3. 前向传播 (Forward Propagation)

### 3.1 什么是前向传播？

前向传播就是数据从输入层到输出层的**计算过程**：

对于每一层：
1. 计算加权和：$z = W \cdot x + b$
2. 通过激活函数：$a = \sigma(z)$
3. 将输出传给下一层

这就像一个**流水线**：每一层接收上一层的输出，计算后传给下一层。

### 3.2 矩阵形式

对于一层有 $m$ 个输入、$n$ 个神经元：

$$\mathbf{z} = \mathbf{W} \cdot \mathbf{x} + \mathbf{b}$$

其中 $\mathbf{W}$ 是 $n \times m$ 的权重矩阵。

In [ ]:
# === 单个神经元的前向传播演示 ===

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    '''Sigmoid的导数：sigma'(x) = sigma(x) * (1 - sigma(x))'''
    s = sigmoid(x)
    return s * (1 - s)

# 单个神经元
inputs = np.array([1.0, 0.5, -0.3])
weights = np.array([0.4, 0.8, -0.2])
bias = 0.1

# 前向传播
z = np.dot(weights, inputs) + bias
output = sigmoid(z)

print('=== 单个神经元前向传播 ===')
print(f'输入 (Inputs):  {inputs}')
print(f'权重 (Weights): {weights}')
print(f'偏置 (Bias):    {bias}')
print()
print('计算过程:')
for i in range(len(inputs)):
    print(f'  w{i+1} * x{i+1} = {weights[i]:.1f} * {inputs[i]:.1f} = {weights[i]*inputs[i]:.2f}')
print(f'  加权和 z = {" + ".join(f"{w*x:.2f}" for w, x in zip(weights, inputs))} + {bias} = {z:.4f}')
print(f'  输出 y = sigmoid({z:.4f}) = {output:.4f}')
print()
print(f'解释：输出 {output:.4f} 接近 {"1" if output > 0.5 else "0"}',


In [ ]:
# === 可视化：前向传播计算过程 ===

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('前向传播计算过程 (Forward Propagation)', fontsize=16, fontweight='bold')

# 定义简单网络: 2输入 -> 2隐藏 -> 1输出
# 输入
inp_pos = [(0.08, 0.7), (0.08, 0.3)]
inp_vals = [1.0, 0.0]
# 隐藏层
hid_pos = [(0.4, 0.75), (0.4, 0.25)]
# 输出层
out_pos = [(0.75, 0.5)]

# 权重（手动设定）
W1 = np.array([[0.5, -0.3], [0.8, 0.2]])  # 隐藏层权重
b1 = np.array([0.1, -0.1])
W2 = np.array([[0.6, -0.4]])  # 输出层权重
b2 = np.array([0.2])

# 计算
x = np.array(inp_vals)
z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
a2 = sigmoid(z2)

# 画输入层
for i, ((px, py), val) in enumerate(zip(inp_pos, inp_vals)):
    circle = plt.Circle((px, py), 0.04, color='#3498DB', ec='black', lw=2, zorder=5)
    ax.add_patch(circle)
    ax.text(px, py, f'x{i+1}\n{val:.1f}', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

# 画隐藏层
for i, (px, py) in enumerate(hid_pos):
    circle = plt.Circle((px, py), 0.05, color='#F39C12', ec='black', lw=2, zorder=5)
    ax.add_patch(circle)
    ax.text(px, py, f'h{i+1}\n{a1[i]:.3f}', ha='center', va='center',
            fontsize=9, fontweight='bold')
    # z值标注
    ax.text(px, py-0.1, f'z={z1[i]:.2f}', ha='center', fontsize=8, color='gray')

# 画输出层
px, py = out_pos[0]
circle = plt.Circle((px, py), 0.05, color='#E74C3C', ec='black', lw=2, zorder=5)
ax.add_patch(circle)
ax.text(px, py, f'y\n{a2[0]:.3f}', ha='center', va='center',
        fontsize=10, fontweight='bold', color='white')
ax.text(px, py-0.1, f'z={z2[0]:.3f}', ha='center', fontsize=8, color='gray')

# 画连接和权重
for i, (ix, iy) in enumerate(inp_pos):
    for j, (hx, hy) in enumerate(hid_pos):
        ax.annotate('', xy=(hx-0.05, hy), xytext=(ix+0.04, iy),
                   arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=1.5))
        mx = (ix + hx) / 2
        my = (iy + hy) / 2
        ax.text(mx, my+0.03, f'w={W1[j][i]:.1f}', fontsize=8, color='#E74C3C',
               ha='center', fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.1', facecolor='lightyellow', alpha=0.9))

for j, (hx, hy) in enumerate(hid_pos):
    ox, oy = out_pos[0]
    ax.annotate('', xy=(ox-0.05, oy), xytext=(hx+0.05, hy),
               arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=1.5))
    mx = (hx + ox) / 2
    my = (hy + oy) / 2
    ax.text(mx, my+0.03, f'w={W2[0][j]:.1f}', fontsize=8, color='#E74C3C',
           ha='center', fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.1', facecolor='lightyellow', alpha=0.9))

# 计算步骤说明
steps_text = [
    f'Step 1: h1 = sigmoid({W1[0][0]:.1f}*{x[0]:.1f} + {W1[0][1]:.1f}*{x[1]:.1f} + {b1[0]:.1f}) = sigmoid({z1[0]:.2f}) = {a1[0]:.3f}',
    f'Step 2: h2 = sigmoid({W1[1][0]:.1f}*{x[0]:.1f} + {W1[1][1]:.1f}*{x[1]:.1f} + {b1[1]:.1f}) = sigmoid({z1[1]:.2f}) = {a1[1]:.3f}',
    f'Step 3: y  = sigmoid({W2[0][0]:.1f}*{a1[0]:.3f} + {W2[0][1]:.1f}*{a1[1]:.3f} + {b2[0]:.1f}) = sigmoid({z2[0]:.3f}) = {a2[0]:.3f}',
]
for i, text in enumerate(steps_text):
    ax.text(0.5, 0.08 - i*0.035, text, ha='center', fontsize=8,
           color='navy', fontfamily='monospace')

plt.tight_layout()
plt.show()
print('前向传播就是逐层计算：加权求和 -> 激活函数 -> 传给下一层')


## 4. 反向传播 (Back Propagation)

### 4.1 核心问题

网络输出和正确答案之间有**误差(Error/Loss)**。怎么调整权重来减小误差？

### 4.2 关键思想：梯度下降 (Gradient Descent)

把Loss想象成一座山，当前权重是山上的一个位置。我们想走到山谷（Loss最小处）。

**梯度(Gradient)** = 山坡的方向和陡度。我们沿着梯度的**反方向**走（下山），就能减小Loss。

$$w_{\text{new}} = w_{\text{old}} - \eta \cdot \frac{\partial \text{Loss}}{\partial w}$$

其中 $\eta$ 是**学习率(Learning Rate)**——每步走多远。

### 4.3 反向传播就是"链式法则"

误差从输出层**向后传播**到每一层，计算每个权重对误差的"贡献度"（偏导数），然后据此调整权重。

利用微积分的**链式法则(Chain Rule)**：

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial z} \cdot \frac{\partial z}{\partial w_1}$$

### 为什么叫"反向"传播？

因为计算顺序和前向传播**相反**：从输出层的误差开始，逐层向后计算每个权重的梯度。

### 学习率的重要性

- **太大**：步子太大，可能跳过最优解，甚至发散
- **太小**：步子太小，收敛非常慢
- 通常从0.01或0.001开始尝试

In [ ]:
# === 可视化：梯度下降过程 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 1D梯度下降 ---
ax = axes[0]
x_gd = np.linspace(-3, 3, 200)
y_gd = x_gd ** 2 + 1  # 简单的抛物线Loss函数
ax.plot(x_gd, y_gd, 'b-', lw=2, label='Loss函数: L(w) = w^2 + 1')

# 梯度下降轨迹
w = 2.5  # 起始点
lr = 0.3
trajectory = [w]
for _ in range(8):
    gradient = 2 * w  # dL/dw = 2w
    w = w - lr * gradient
    trajectory.append(w)

for i, w_val in enumerate(trajectory):
    l_val = w_val**2 + 1
    color = plt.cm.Reds(0.3 + 0.7 * i / len(trajectory))
    ax.plot(w_val, l_val, 'o', color=color, markersize=8, zorder=5)
    if i < len(trajectory) - 1:
        w_next = trajectory[i+1]
        l_next = w_next**2 + 1
        ax.annotate('', xy=(w_next, l_next), xytext=(w_val, l_val),
                   arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

ax.set_title('梯度下降 (lr=0.3)', fontsize=13, fontweight='bold')
ax.set_xlabel('权重 w')
ax.set_ylabel('Loss')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- 学习率对比 ---
ax = axes[1]
for lr_val, color, ls in [(0.05, '#3498DB', '-'), (0.3, '#2ECC71', '-'), (0.9, '#E74C3C', '-')]:
    w = 2.5
    losses = [w**2 + 1]
    for _ in range(15):
        w = w - lr_val * 2 * w
        losses.append(w**2 + 1)
    ax.plot(losses, color=color, lw=2, ls=ls, label=f'lr={lr_val}')

ax.set_title('学习率对收敛的影响', fontsize=13, fontweight='bold')
ax.set_xlabel('迭代次数 (Epoch)')
ax.set_ylabel('Loss')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 10)

# --- 2D等高线图 ---
ax = axes[2]
w1_range = np.linspace(-3, 3, 100)
w2_range = np.linspace(-3, 3, 100)
W1_grid, W2_grid = np.meshgrid(w1_range, w2_range)
Loss_grid = W1_grid**2 + W2_grid**2 + 1

ax.contour(W1_grid, W2_grid, Loss_grid, levels=15, cmap='viridis', alpha=0.7)
ax.contourf(W1_grid, W2_grid, Loss_grid, levels=15, cmap='viridis', alpha=0.3)

# 轨迹
w1, w2 = 2.5, -2.0
lr_2d = 0.2
traj_w1, traj_w2 = [w1], [w2]
for _ in range(12):
    w1 = w1 - lr_2d * 2 * w1
    w2 = w2 - lr_2d * 2 * w2
    traj_w1.append(w1)
    traj_w2.append(w2)

ax.plot(traj_w1, traj_w2, 'ro-', markersize=5, lw=1.5, label='梯度下降轨迹')
ax.plot(0, 0, 'g*', markersize=15, label='最优解')
ax.set_title('2D梯度下降', fontsize=13, fontweight='bold')
ax.set_xlabel('w1')
ax.set_ylabel('w2')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('左图：梯度下降沿着Loss函数的斜坡"滑下"到最低点。')
print('中图：学习率太小收敛慢，太大会震荡甚至发散。')


## 5. 实战：从零构建神经网络解决XOR问题

### 5.1 为什么是XOR？

XOR (异或) 是经典的**非线性**问题：

| 输入1 | 输入2 | XOR输出 |
|-------|-------|--------|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

**为什么XOR特别？** 单层感知器（没有隐藏层）**无法**解决XOR——因为XOR不是线性可分的。这曾经是1969年Minsky和Papert提出的著名挑战，导致了第一次"AI寒冬"。

但加入隐藏层后，神经网络就能解决它！这证明了隐藏层的必要性。

In [ ]:
# === 可视化：为什么XOR线性不可分 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# AND - 线性可分
ax = axes[0]
ax.scatter([0, 0, 1], [0, 1, 0], c='#E74C3C', s=150, marker='x', lw=3, label='0', zorder=5)
ax.scatter([1], [1], c='#2ECC71', s=150, marker='o', lw=3, label='1', zorder=5)
x_line = np.linspace(-0.5, 1.5, 100)
ax.plot(x_line, -x_line + 1.5, 'b--', lw=2, label='分界线')
ax.fill_between(x_line, -x_line + 1.5, 2, alpha=0.1, color='green')
ax.fill_between(x_line, -1, -x_line + 1.5, alpha=0.1, color='red')
ax.set_title('AND: 线性可分', fontsize=13, fontweight='bold')
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlabel('x1'); ax.set_ylabel('x2')

# OR - 线性可分
ax = axes[1]
ax.scatter([0], [0], c='#E74C3C', s=150, marker='x', lw=3, label='0', zorder=5)
ax.scatter([0, 1, 1], [1, 0, 1], c='#2ECC71', s=150, marker='o', lw=3, label='1', zorder=5)
ax.plot(x_line, -x_line + 0.5, 'b--', lw=2, label='分界线')
ax.fill_between(x_line, -x_line + 0.5, 2, alpha=0.1, color='green')
ax.fill_between(x_line, -1, -x_line + 0.5, alpha=0.1, color='red')
ax.set_title('OR: 线性可分', fontsize=13, fontweight='bold')
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlabel('x1'); ax.set_ylabel('x2')

# XOR - 线性不可分
ax = axes[2]
ax.scatter([0, 1], [0, 1], c='#E74C3C', s=150, marker='x', lw=3, label='0', zorder=5)
ax.scatter([0, 1], [1, 0], c='#2ECC71', s=150, marker='o', lw=3, label='1', zorder=5)
ax.set_title('XOR: 线性不可分!', fontsize=13, fontweight='bold', color='red')
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.text(0.5, -0.3, '找不到一条直线分开!', ha='center', fontsize=11,
        color='red', fontweight='bold')

plt.tight_layout()
plt.show()
print('AND和OR可以用一条直线分开两类点，但XOR不行！')


In [ ]:
# === 从零构建神经网络：解决XOR问题 ===

class NeuralNetwork:
    '''
    简单的2层神经网络（1个隐藏层）
    结构: 2输入 -> 4隐藏 -> 1输出
    '''

    def __init__(self, input_size=2, hidden_size=4, output_size=1, learning_rate=0.5):
        self.lr = learning_rate
        # 随机初始化权重（使用小随机数）
        self.W1 = np.random.randn(input_size, hidden_size) * 0.5
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.5
        self.b2 = np.zeros((1, output_size))
        self.losses = []  # 记录训练过程中的损失

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def sigmoid_deriv(self, x):
        s = self.sigmoid(x)
        return s * (1 - s)

    def forward(self, X):
        '''前向传播'''
        self.z1 = X @ self.W1 + self.b1       # 隐藏层加权和
        self.a1 = self.sigmoid(self.z1)         # 隐藏层输出
        self.z2 = self.a1 @ self.W2 + self.b2   # 输出层加权和
        self.a2 = self.sigmoid(self.z2)          # 输出层输出（预测值）
        return self.a2

    def backward(self, X, y):
        '''反向传播：计算梯度并更新权重'''
        m = len(X)  # 样本数

        # 输出层误差
        dz2 = self.a2 - y                               # (m, 1)
        dW2 = self.a1.T @ dz2 / m                       # (hidden, 1)
        db2 = np.sum(dz2, axis=0, keepdims=True) / m

        # 隐藏层误差（通过链式法则向后传播）
        dz1 = (dz2 @ self.W2.T) * self.sigmoid_deriv(self.z1)  # (m, hidden)
        dW1 = X.T @ dz1 / m                                      # (input, hidden)
        db1 = np.sum(dz1, axis=0, keepdims=True) / m

        # 更新权重（梯度下降）
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train(self, X, y, epochs=10000, print_every=2000):
        '''训练网络'''
        for epoch in range(epochs):
            # 前向传播
            output = self.forward(X)
            # 计算损失（均方误差）
            loss = np.mean((y - output) ** 2)
            self.losses.append(loss)
            # 反向传播
            self.backward(X, y)

            if (epoch + 1) % print_every == 0:
                print(f'Epoch {epoch+1:>5}, Loss: {loss:.6f}')

        print(f'\n训练完成! 最终Loss: {self.losses[-1]:.6f}')


# XOR数据
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([[0], [1], [1], [0]])

# 创建并训练网络
print('=== 训练神经网络解决XOR问题 ===')
print(f'网络结构: 2 -> 4 -> 1')
print(f'学习率: 2.0')
print()

nn = NeuralNetwork(input_size=2, hidden_size=4, output_size=1, learning_rate=2.0)
nn.train(X_xor, y_xor, epochs=10000, print_every=2000)

# 测试
print('\n=== 测试结果 ===')
predictions = nn.forward(X_xor)
for i in range(len(X_xor)):
    pred_val = predictions[i][0]
    pred_class = 1 if pred_val > 0.5 else 0
    correct = 'Correct' if pred_class == y_xor[i][0] else 'WRONG'


In [ ]:
# === 可视化：训练过程和决策边界 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 损失函数变化 ---
ax = axes[0]
ax.plot(nn.losses, color='#E74C3C', lw=1.5)
ax.set_title('训练过程中Loss的变化', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch (迭代次数)')
ax.set_ylabel('Mean Squared Error (MSE)')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.axhline(y=0.01, color='green', ls='--', alpha=0.5, label='目标Loss')
ax.legend()

# --- 决策边界 ---
ax = axes[1]
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid_input = np.c_[xx.ravel(), yy.ravel()]
Z = nn.forward(grid_input).reshape(xx.shape)

ax.contourf(xx, yy, Z, levels=50, cmap='RdYlGn', alpha=0.7)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

# 数据点
for i in range(4):
    color = '#2ECC71' if y_xor[i][0] == 1 else '#E74C3C'
    marker = 'o' if y_xor[i][0] == 1 else 'x'
    ax.scatter(X_xor[i][0], X_xor[i][1], c=color, s=200, marker=marker,
              edgecolors='black', lw=2, zorder=5)
    ax.text(X_xor[i][0]+0.08, X_xor[i][1]+0.08,
           f'({X_xor[i][0]},{X_xor[i][1]})={y_xor[i][0]}',
           fontsize=10, fontweight='bold')

ax.set_title('XOR决策边界', fontsize=13, fontweight='bold')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
cb = plt.colorbar(ax.contourf(xx, yy, Z, levels=50, cmap='RdYlGn', alpha=0), ax=ax)
cb.set_label('网络输出值')

# --- 隐藏层的"变换" ---
ax = axes[2]
# 显示隐藏层如何变换输入空间
hidden_outputs = nn.sigmoid(X_xor @ nn.W1 + nn.b1)
# 取前两个隐藏神经元的输出来可视化
for i in range(4):
    color = '#2ECC71' if y_xor[i][0] == 1 else '#E74C3C'
    marker = 'o' if y_xor[i][0] == 1 else 'x'
    ax.scatter(hidden_outputs[i][0], hidden_outputs[i][1],
              c=color, s=200, marker=marker, edgecolors='black', lw=2, zorder=5)
    ax.text(hidden_outputs[i][0]+0.02, hidden_outputs[i][1]+0.02,
           f'({X_xor[i][0]},{X_xor[i][1]})', fontsize=10)

ax.set_title('隐藏层变换后的空间\n(前2个神经元)', fontsize=13, fontweight='bold')
ax.set_xlabel('隐藏神经元1的输出')
ax.set_ylabel('隐藏神经元2的输出')
ax.grid(True, alpha=0.3)

# 尝试画分界线
h_xx, h_yy = np.meshgrid(np.linspace(0, 1, 50), np.linspace(0, 1, 50))
# 简单显示
ax.text(0.5, 0.05, '隐藏层将数据变换到新空间，\n使其变得线性可分！',
       ha='center', fontsize=10, color='navy', fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.show()
print('左图：Loss随训练迭代逐渐下降，网络在"学习"。')
print('中图：网络学会了一个非线性的决策边界来正确分类XOR。')


## 6. 深度学习概念

### 6.1 什么是"深度"？

深度学习 = 使用**多层隐藏层**的神经网络。"深度"指的是网络的层数多。

现代深度网络可以有几十甚至几百层（如ResNet-152有152层）。

### 6.2 卷积神经网络 CNN (Convolutional Neural Network)

CNN是专门处理**图像**的深度学习架构：

- **卷积层(Convolutional Layer)**：使用"滤波器"扫描图像，自动提取特征
  - 低层特征：边缘、纹理
  - 中层特征：眼睛、耳朵等部件
  - 高层特征：人脸、汽车等整体
- **池化层(Pooling Layer)**：缩小数据维度，保留重要信息
- **全连接层**：最后做分类决策

### 6.3 为什么CNN对图像特别有效？

1. **参数共享**：同一个滤波器扫描整张图，参数少得多
2. **平移不变性**：猫在图片左边还是右边，都能识别
3. **层次化特征学习**：自动从简单到复杂地学习特征

In [ ]:
# === 可视化：CNN的层次化特征学习 ===

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('CNN: 从像素到识别的层次化学习', fontsize=16, fontweight='bold')

# 画各层
layers = [
    {'x': 0.05, 'w': 0.12, 'h': 0.7, 'color': '#3498DB', 'label': 'Input\n输入图像',
     'detail': '原始像素\n224x224x3'},
    {'x': 0.22, 'w': 0.10, 'h': 0.6, 'color': '#2ECC71', 'label': 'Conv1\n卷积层1',
     'detail': '边缘/纹理\n检测'},
    {'x': 0.35, 'w': 0.08, 'h': 0.5, 'color': '#F39C12', 'label': 'Conv2\n卷积层2',
     'detail': '局部特征\n(眼/鼻)'},
    {'x': 0.46, 'w': 0.06, 'h': 0.4, 'color': '#E74C3C', 'label': 'Conv3\n卷积层3',
     'detail': '复杂特征\n(人脸)'},
    {'x': 0.56, 'w': 0.04, 'h': 0.3, 'color': '#9B59B6', 'label': 'Pool\n池化层',
     'detail': '降维\n压缩'},
    {'x': 0.66, 'w': 0.12, 'h': 0.15, 'color': '#1ABC9C', 'label': 'FC\n全连接层',
     'detail': '综合判断'},
    {'x': 0.84, 'w': 0.10, 'h': 0.08, 'color': '#E67E22', 'label': 'Output\n输出',
     'detail': '猫: 95%\n狗: 3%\n鸟: 2%'},
]

for layer in layers:
    y = 0.5 - layer['h'] / 2
    rect = plt.Rectangle((layer['x'], y), layer['w'], layer['h'],
                         facecolor=layer['color'], alpha=0.7,
                         edgecolor='black', lw=2)
    ax.add_patch(rect)
    ax.text(layer['x'] + layer['w']/2, y + layer['h'] + 0.03,
           layer['label'], ha='center', fontsize=9, fontweight='bold')
    ax.text(layer['x'] + layer['w']/2, y + layer['h']/2,
           layer['detail'], ha='center', va='center', fontsize=7,
           color='white', fontweight='bold')

# 箭头
arrow_positions = [(0.17, 0.22), (0.32, 0.35), (0.43, 0.46),
                   (0.52, 0.56), (0.60, 0.66), (0.78, 0.84)]
for x1, x2 in arrow_positions:
    ax.annotate('', xy=(x2, 0.5), xytext=(x1, 0.5),
               arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# 底部说明
ax.text(0.5, 0.06,
       '每一层提取的特征越来越抽象：像素 -> 边缘 -> 部件 -> 整体对象',
       ha='center', fontsize=12, color='navy', fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))

plt.tight_layout()
plt.show()


In [ ]:
# === 可视化：卷积操作演示 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 简单的5x5图像
image = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 0, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
], dtype=float)

# 3x3边缘检测滤波器
kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=float)

# 手动卷积
output_size = image.shape[0] - kernel.shape[0] + 1
conv_output = np.zeros((output_size, output_size))
for i in range(output_size):
    for j in range(output_size):
        region = image[i:i+3, j:j+3]
        conv_output[i, j] = np.sum(region * kernel)

# 画原始图像
ax = axes[0]
im1 = ax.imshow(image, cmap='gray', interpolation='nearest')
ax.set_title('原始图像 (5x5)', fontsize=13, fontweight='bold')
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{image[i,j]:.0f}', ha='center', va='center',
               fontsize=14, color='red' if image[i,j] == 0 else 'blue', fontweight='bold')

# 画滤波器
ax = axes[1]
im2 = ax.imshow(kernel, cmap='RdBu', interpolation='nearest', vmin=-8, vmax=8)
ax.set_title('边缘检测滤波器 (3x3)', fontsize=13, fontweight='bold')
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{kernel[i,j]:.0f}', ha='center', va='center',
               fontsize=16, fontweight='bold',
               color='white' if abs(kernel[i,j]) > 3 else 'black')
plt.colorbar(im2, ax=ax)

# 画卷积输出
ax = axes[2]
im3 = ax.imshow(conv_output, cmap='hot', interpolation='nearest')
ax.set_title('卷积输出 (3x3)', fontsize=13, fontweight='bold')
for i in range(output_size):
    for j in range(output_size):
        ax.text(j, i, f'{conv_output[i,j]:.0f}', ha='center', va='center',
               fontsize=16, fontweight='bold', color='white')
plt.colorbar(im3, ax=ax)

plt.suptitle('卷积操作 (Convolution Operation)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('卷积 = 滤波器在图像上滑动，每个位置做元素乘法再求和。')
print('这个边缘检测滤波器会在边缘处产生高值（亮色），在平坦区域产生0。')


## 7. 深入思考：ChatGPT和图像识别的技术原理

### 7.1 ChatGPT背后：Transformer架构

ChatGPT基于**Transformer**架构的**大语言模型(LLM)**：

- **核心机制：自注意力(Self-Attention)**
  - 让模型关注输入序列中**任意位置**的关系
  - 比如理解"The cat sat on the mat because **it** was tired"中"it"指代"cat"

- **训练方式**：
  1. **预训练**：在海量文本上学习语言模式（预测下一个词）
  2. **微调**：在特定任务上调整（如对话）
  3. **RLHF**：用人类反馈进一步对齐

- **规模**：GPT-4有**数千亿**参数，在**TB级**文本上训练

### 7.2 图像识别：从AlexNet到现代模型

- **2012 AlexNet**：8层CNN，让深度学习一战成名
- **2015 ResNet**：152层！引入"残差连接"解决深层网络训练难题
- **2020+ Vision Transformer (ViT)**：将Transformer用于图像，效果更好

### 为什么这些技术在2010年后才爆发？

三个关键因素同时成熟：
1. **数据**：互联网产生了海量数据（ImageNet, Wikipedia等）
2. **算力**：GPU让并行计算快了100倍
3. **算法**：反向传播、Dropout、BatchNorm等关键技巧

---

## 8. 练习题 (Exercises)

### 练习1：概念题

1. 一个输入层3个神经元、隐藏层5个神经元、输出层2个神经元的全连接网络有多少个权重参数？多少个偏置参数？总共多少个可训练参数？

2. 解释为什么激活函数必须是非线性的。如果所有激活函数都是线性的 $f(x) = kx$，三层网络会退化为什么？

3. 学习率(Learning Rate)设置为0.0001和10.0分别会导致什么问题？

4. 为什么CNN比全连接网络更适合处理图像？（从参数数量和特征提取两个角度分析）

### 练习2：手动前向传播

In [ ]:
# === 练习2：手动计算前向传播 ===

# 给定一个简单网络:
# 输入: [0.5, 0.8]
# 隐藏层权重: W1 = [[0.3, 0.7], [-0.2, 0.5]]
# 隐藏层偏置: b1 = [0.1, -0.1]
# 输出层权重: W2 = [[0.6], [-0.3]]
# 输出层偏置: b2 = [0.2]
# 激活函数: sigmoid

# 请手动计算:
# 1. 隐藏层的加权和 z1
# 2. 隐藏层的输出 a1 (通过sigmoid)
# 3. 输出层的加权和 z2
# 4. 最终输出 a2

print('请先在纸上计算，然后运行下一个cell验证。')
print()
print('提示:')
print('  z1[0] = W1[0][0]*x[0] + W1[0][1]*x[1] + b1[0]')


In [ ]:
# === 练习2答案 ===

# 取消注释查看
# x = np.array([0.5, 0.8])
# W1 = np.array([[0.3, 0.7], [-0.2, 0.5]])
# b1 = np.array([0.1, -0.1])
# W2 = np.array([[0.6], [-0.3]])
# b2 = np.array([0.2])
#
# z1 = x @ W1 + b1
# print(f'z1 = {z1}')
# a1 = 1 / (1 + np.exp(-z1))
# print(f'a1 = sigmoid(z1) = {a1}')
# z2 = a1 @ W2 + b2
# print(f'z2 = {z2}')
# a2 = 1 / (1 + np.exp(-z2))
# print(f'a2 = sigmoid(z2) = {a2}')


In [ ]:
# === 练习3（拓展）：修改XOR网络 ===

# 尝试修改以下参数，观察对训练的影响：
# 1. 改变隐藏层神经元数量（试试2个够不够？需要几个？）
# 2. 改变学习率（0.1 vs 1.0 vs 5.0）
# 3. 改变训练轮数

print('=== 实验：隐藏层大小的影响 ===')
for hidden in [1, 2, 4, 8]:
    nn_test = NeuralNetwork(input_size=2, hidden_size=hidden, output_size=1, learning_rate=2.0)
    nn_test.train(X_xor, y_xor, epochs=10000, print_every=100000)  # 静默训练
    preds = nn_test.forward(X_xor)
    correct = sum(1 for i in range(4) if round(preds[i][0]) == y_xor[i][0])
    print(f'  隐藏层={hidden}: 最终Loss={nn_test.losses[-1]:.6f}, 正确率={correct}/4')

print()
print('实验结论：隐藏层至少需要2个神经元才能解决XOR（因为需要2条线）。')


In [ ]:
# === 练习4（挑战）：用神经网络学习更复杂的模式 ===

# 生成环形数据（线性不可分）
np.random.seed(42)
n_samples = 200
theta = np.random.uniform(0, 2*np.pi, n_samples)
r_inner = np.random.uniform(0, 1, n_samples // 2)
r_outer = np.random.uniform(2, 3, n_samples // 2)

X_circle = np.vstack([
    np.column_stack([r_inner * np.cos(theta[:n_samples//2]),
                     r_inner * np.sin(theta[:n_samples//2])]),
    np.column_stack([r_outer * np.cos(theta[n_samples//2:]),
                     r_outer * np.sin(theta[n_samples//2:])])
])
y_circle = np.vstack([np.zeros((n_samples//2, 1)), np.ones((n_samples//2, 1))])

# 可视化数据
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

mask0 = y_circle.ravel() == 0
mask1 = y_circle.ravel() == 1
ax1.scatter(X_circle[mask0, 0], X_circle[mask0, 1], c='#E74C3C', s=20, label='Class 0 (内环)')
ax1.scatter(X_circle[mask1, 0], X_circle[mask1, 1], c='#3498DB', s=20, label='Class 1 (外环)')
ax1.set_title('环形数据 - 线性不可分!', fontsize=13, fontweight='bold')
ax1.set_aspect('equal')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 训练网络
nn_circle = NeuralNetwork(input_size=2, hidden_size=8, output_size=1, learning_rate=1.0)
nn_circle.train(X_circle, y_circle, epochs=5000, print_every=1000)

# 决策边界
xx, yy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = nn_circle.forward(grid).reshape(xx.shape)

ax2.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.5)
ax2.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
ax2.scatter(X_circle[mask0, 0], X_circle[mask0, 1], c='#E74C3C', s=20)
ax2.scatter(X_circle[mask1, 0], X_circle[mask1, 1], c='#3498DB', s=20)
ax2.set_title('神经网络的非线性决策边界', fontsize=13, fontweight='bold')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('神经网络能学习出环形的决策边界——这是线性模型做不到的！')


---

## 本节小结 (Summary)

| 概念 | 要点 |
|------|------|
| 人工神经元 | 加权求和 + 偏置 + 激活函数 |
| 激活函数 | Sigmoid、ReLU等，引入非线性 |
| 网络结构 | 输入层 -> 隐藏层(可多个) -> 输出层 |
| 前向传播 | 从输入到输出的逐层计算 |
| 反向传播 | 从输出到输入计算梯度，用梯度下降更新权重 |
| 学习率 | 控制每步更新的大小，太大/太小都有问题 |
| 深度学习 | 多层网络，能自动学习层次化特征 |
| CNN | 卷积+池化，专门处理图像 |

### 考试关键点

1. 能画出并解释ANN的结构
2. 能手动计算前向传播
3. 理解反向传播的**思想**（不需要推导数学）
4. 知道为什么需要激活函数（非线性）
5. 知道CNN用于图像识别的基本原理

---

**恭喜你完成了第18章的全部内容！** 从图搜索算法到机器学习再到神经网络，你已经了解了AI的核心技术栈。